In [1]:
import os
import json
from pathlib import Path

def read_folder_structure(root_path):
    """
    بيقرأ كل الفولدرات والملفات وبيرجع structure كاملة
    """
    structure = {}
    
    for dirpath, dirnames, filenames in os.walk(root_path):
        # المسار النسبي من الـ root
        relative_path = os.path.relpath(dirpath, root_path)
        
        # لو في ملفات في الفولدر ده
        if filenames:
            structure[relative_path] = filenames
    
    return structure


def save_structure(structure, output_file="folder_structure.txt"):
    """
    بيحفظ الـ structure في ملف txt واضح
    """
    with open(output_file, "w", encoding="utf-8") as f:
        
        f.write("=" * 60 + "\n")
        f.write("       PROJECT FOLDER STRUCTURE\n")
        f.write("=" * 60 + "\n\n")
        
        for folder, files in structure.items():
            # اسم الفولدر
            f.write(f"📁 FOLDER: {folder}\n")
            f.write("-" * 40 + "\n")
            
            # الملفات جوا الفولدر
            for file in files:
                f.write(f"    📄 {file}\n")
            
            f.write("\n")
    
    print(f"✅ تم الحفظ في: {output_file}")


def print_structure(structure):
    """
    بيطبع الـ structure على الشاشة
    """
    print("=" * 60)
    print("       PROJECT FOLDER STRUCTURE")
    print("=" * 60)
    
    for folder, files in structure.items():
        print(f"\n📁 FOLDER: {folder}")
        print("-" * 40)
        for file in files:
            print(f"    📄 {file}")


# ============================================
# 🔴 غيّر المسار ده لمسار الفولدر بتاعك
# ============================================
root_path = r"WFM"

# تشغيل الكود
structure = read_folder_structure(root_path)

# طباعة على الشاشة
print_structure(structure)

# حفظ في ملف
save_structure(structure, "folder_structure.txt")

# حفظ في JSON كمان (اختياري)
with open("folder_structure.json", "w", encoding="utf-8") as f:
    json.dump(structure, f, ensure_ascii=False, indent=4)
    print("✅ تم الحفظ في: folder_structure.json")

       PROJECT FOLDER STRUCTURE

📁 FOLDER: (-2025-200)FCC Operations Monitor Dashboard Batch 2 CR (v0.1)\CR Document
----------------------------------------
    📄 FCC Operations Monitoring Dashboard Batch 2 (v0.1) (2).docx

📁 FOLDER: (-2025-201)WFM Operations Monitor Dashboard Batch 2 CR (v0.1)\CR Document
----------------------------------------
    📄 WFM Operations Monitor Dashboard Batch 2 CR (v0.1) (1).docx

📁 FOLDER: (CR-2024-136)WFM Box Complaints CR\CR Documents
----------------------------------------
    📄 (CR-2024-136)WFM BOX Complain CR (V0.1).docx

📁 FOLDER: (CR-2024-156) WFM - SA Post Production - Modify SA APIs CR v3
----------------------------------------
    📄 WFM - SA Post Production - Modify SA APIs CR v3 - Workload (2).docx
    📄 WFM - SA Post Production - Modify SA APIs CR v3 -.docx

📁 FOLDER: (CR-2024-170) WFM Detect Enterprise Work Orders CR\CR document
----------------------------------------
    📄 (CR-2024-170) WFM Detect Enterprise Work Orders CR(V0.1) - Work

In [3]:
import os
import json
from pathlib import Path

def clean_and_analyze(root_path):
    """
    بيحلل الملفات ويعمل تصنيف نظيف
    """
    
    # =========================================
    # الملفات اللي هنتجاهلها
    # =========================================
    IGNORE_PATTERNS = [
        "~$",           # temp files
        "desktop.ini",  # windows files
        ".svn",         # version control
        "- Workload",   # workload files
        "+WL",          # workload files
        "+wl",          # workload files
        ".svn-base"     # svn base files
    ]
    
    VALID_EXTENSIONS = ['.docx', '.xlsx', '.pdf']
    
    # =========================================
    # تصنيف الـ CRs حسب النوع
    # =========================================
    CATEGORIES = {
        "Scheduling": [
            "Scheduling", "Schedule", "Time Slot", 
            "Time Interval", "Automatic Scheduling"
        ],
        "Maintenance": [
            "Maintenance", "FTTH Maintenance", 
            "Technical Maintenance", "BPD"
        ],
        "Installation": [
            "Installation", "Reserve Activation", 
            "Migration", "Survey", "FTTH Migration"
        ],
        "Service_Activation": [
            "SA", "Service Activation", "cross-connection",
            "Wrong Card", "Port"
        ],
        "Dashboard_Reports": [
            "Dashboard", "KPI", "Reports", "Reporting",
            "Monitoring", "Performance"
        ],
        "Security": [
            "Security", "Data Leakage", "Authentication",
            "Quota", "Session"
        ],
        "User_Management": [
            "Login", "User", "Desktop Users", 
            "Team Leader", "Mobile"
        ],
        "Device_Management": [
            "CPE", "Devices", "Installment", 
            "Sold Devices", "Rent"
        ],
        "Other": []
    }
    
    results = {
        "clean_files": [],      # الملفات النظيفة
        "ignored_files": [],    # الملفات اللي اتشالت
        "categories": {},       # تصنيف الملفات
        "duplicates": [],       # ملفات متكررة
        "stats": {}             # إحصائيات
    }
    
    # تهيئة الـ categories
    for cat in CATEGORIES:
        results["categories"][cat] = []
    
    seen_names = {}  # عشان نكتشف التكرار
    
    # =========================================
    # المشي على كل الملفات
    # =========================================
    for dirpath, dirnames, filenames in os.walk(root_path):
        
        # تجاهل فولدرات .svn
        dirnames[:] = [d for d in dirnames if d != '.svn']
        
        relative_path = os.path.relpath(dirpath, root_path)
        
        for filename in filenames:
            full_path = os.path.join(dirpath, filename)
            
            # =====================================
            # هل الملف يتشال؟
            # =====================================
            should_ignore = False
            ignore_reason = ""
            
            for pattern in IGNORE_PATTERNS:
                if pattern.lower() in filename.lower() or \
                   pattern.lower() in relative_path.lower():
                    should_ignore = True
                    ignore_reason = f"Pattern: {pattern}"
                    break
            
            # تجاهل الامتدادات الغلط
            file_ext = Path(filename).suffix.lower()
            if file_ext not in VALID_EXTENSIONS:
                should_ignore = True
                ignore_reason = f"Extension: {file_ext}"
            
            if should_ignore:
                results["ignored_files"].append({
                    "file": filename,
                    "path": relative_path,
                    "reason": ignore_reason
                })
                continue
            
            # =====================================
            # الملف نظيف - صنّفه
            # =====================================
            file_info = {
                "filename": filename,
                "folder": relative_path,
                "extension": file_ext,
                "type": get_doc_type(filename),
                "cr_number": extract_cr_number(filename, relative_path),
                "version": extract_version(filename),
                "category": categorize(filename, relative_path, CATEGORIES)
            }
            
            results["clean_files"].append(file_info)
            
            # تصنيف في الـ category
            cat = file_info["category"]
            results["categories"][cat].append(file_info)
            
            # كشف التكرار
            base_name = Path(filename).stem.lower()
            if base_name in seen_names:
                results["duplicates"].append({
                    "file": filename,
                    "original": seen_names[base_name],
                    "current_path": relative_path
                })
            else:
                seen_names[base_name] = relative_path
    
    # =====================================
    # إحصائيات
    # =====================================
    results["stats"] = {
        "total_clean_files": len(results["clean_files"]),
        "total_ignored": len(results["ignored_files"]),
        "total_duplicates": len(results["duplicates"]),
        "by_category": {
            cat: len(files) 
            for cat, files in results["categories"].items()
        },
        "by_extension": {
            ".docx": len([f for f in results["clean_files"] 
                         if f["extension"] == ".docx"]),
            ".xlsx": len([f for f in results["clean_files"] 
                         if f["extension"] == ".xlsx"]),
            ".pdf": len([f for f in results["clean_files"] 
                        if f["extension"] == ".pdf"]),
        }
    }
    
    return results


def get_doc_type(filename):
    """تحديد نوع الوثيقة"""
    filename_lower = filename.lower()
    
    if "atp" in filename_lower:
        return "ATP"  # Acceptance Test Plan
    elif "tech" in filename_lower or "technical" in filename_lower:
        return "Technical_Doc"
    elif "e2e" in filename_lower:
        return "E2E_Plan"
    elif "iot" in filename_lower:
        return "IOT_Plan"
    elif "frs" in filename_lower or "bpd" in filename_lower:
        return "FRS_BPD"
    elif "manual" in filename_lower:
        return "User_Manual"
    elif "cr" in filename_lower:
        return "CR_Document"
    else:
        return "Other"


def extract_cr_number(filename, folder_path):
    """استخراج رقم الـ CR"""
    import re
    
    # دور على pattern زي CR-2025-220 أو PM-2024-168
    pattern = r'(CR|PM)-(\d{4})-(\d+)'
    
    # ابحث في اسم الفولدر الأول
    match = re.search(pattern, folder_path, re.IGNORECASE)
    if match:
        return match.group(0)
    
    # ابحث في اسم الملف
    match = re.search(pattern, filename, re.IGNORECASE)
    if match:
        return match.group(0)
    
    return "Unknown"


def extract_version(filename):
    """استخراج رقم الـ Version"""
    import re
    
    # دور على v0.2 أو V3.5 أو (v1.0)
    pattern = r'[Vv](\d+\.\d+)'
    match = re.search(pattern, filename)
    if match:
        return match.group(0)
    
    return "Unknown"


def categorize(filename, folder_path, categories):
    """تصنيف الملف"""
    text = (filename + " " + folder_path).lower()
    
    for category, keywords in categories.items():
        if category == "Other":
            continue
        for keyword in keywords:
            if keyword.lower() in text:
                return category
    
    return "Other"


def save_analysis(results, output_path="analysis"):
    """حفظ النتائج"""
    os.makedirs(output_path, exist_ok=True)
    
    # 1. الملفات النظيفة
    with open(f"{output_path}/clean_files.json", "w", 
              encoding="utf-8") as f:
        json.dump(results["clean_files"], f, 
                  ensure_ascii=False, indent=2)
    
    # 2. الإحصائيات
    with open(f"{output_path}/stats.json", "w", 
              encoding="utf-8") as f:
        json.dump(results["stats"], f, 
                  ensure_ascii=False, indent=2)
    
    # 3. التصنيفات
    with open(f"{output_path}/categories.json", "w", 
              encoding="utf-8") as f:
        json.dump(results["categories"], f, 
                  ensure_ascii=False, indent=2)
    
    # 4. تقرير نصي واضح
    with open(f"{output_path}/report.txt", "w", 
              encoding="utf-8") as f:
        
        f.write("=" * 60 + "\n")
        f.write("          WFM PROJECT ANALYSIS REPORT\n")
        f.write("=" * 60 + "\n\n")
        
        # إحصائيات عامة
        f.write("📊 STATISTICS:\n")
        f.write("-" * 40 + "\n")
        f.write(f"✅ Clean Files: {results['stats']['total_clean_files']}\n")
        f.write(f"❌ Ignored Files: {results['stats']['total_ignored']}\n")
        f.write(f"⚠️  Duplicates: {results['stats']['total_duplicates']}\n\n")
        
        # حسب الامتداد
        f.write("📄 BY EXTENSION:\n")
        f.write("-" * 40 + "\n")
        for ext, count in results['stats']['by_extension'].items():
            f.write(f"   {ext}: {count} files\n")
        f.write("\n")
        
        # حسب التصنيف
        f.write("🏷️ BY CATEGORY:\n")
        f.write("-" * 40 + "\n")
        for cat, count in results['stats']['by_category'].items():
            if count > 0:
                f.write(f"   {cat}: {count} files\n")
        f.write("\n")
        
        # تفاصيل كل category
        f.write("=" * 60 + "\n")
        f.write("          DETAILED CATEGORIES\n")
        f.write("=" * 60 + "\n\n")
        
        for category, files in results["categories"].items():
            if not files:
                continue
            f.write(f"\n🏷️ {category} ({len(files)} files):\n")
            f.write("-" * 40 + "\n")
            for file in files:
                f.write(f"   📄 {file['filename']}\n")
                f.write(f"      CR#: {file['cr_number']}\n")
                f.write(f"      Type: {file['type']}\n")
                f.write(f"      Version: {file['version']}\n")
                f.write(f"      Folder: {file['folder']}\n\n")
    
    print("✅ Analysis saved in:", output_path)
    print(f"   - clean_files.json")
    print(f"   - stats.json")
    print(f"   - categories.json")
    print(f"   - report.txt")


# =========================================
# تشغيل الكود
# =========================================
root_path = r"WFM"

print("🔍 Analyzing folder structure...")
results = clean_and_analyze(root_path)

# طباعة الإحصائيات
print("\n" + "="*50)
print("📊 QUICK STATS:")
print("="*50)
print(f"✅ Clean files: {results['stats']['total_clean_files']}")
print(f"❌ Ignored files: {results['stats']['total_ignored']}")
print(f"⚠️  Duplicates: {results['stats']['total_duplicates']}")
print("\n🏷️ By Category:")
for cat, count in results['stats']['by_category'].items():
    if count > 0:
        print(f"   {cat}: {count} files")

# حفظ النتائج
save_analysis(results)

🔍 Analyzing folder structure...

📊 QUICK STATS:
✅ Clean files: 236
❌ Ignored files: 96
⚠️  Duplicates: 56

🏷️ By Category:
   Scheduling: 75 files
   Maintenance: 31 files
   Installation: 30 files
   Service_Activation: 36 files
   Dashboard_Reports: 12 files
   User_Management: 12 files
   Device_Management: 9 files
   Other: 31 files
✅ Analysis saved in: analysis
   - clean_files.json
   - stats.json
   - categories.json
   - report.txt


In [4]:
import os
import json
import re
import shutil
from pathlib import Path
from collections import defaultdict

# ============================================
# 1️⃣ لود الملفات من التحليل السابق
# ============================================
with open("analysis/clean_files.json", "r", encoding="utf-8") as f:
    clean_files = json.load(f)

print(f"📂 Loaded: {len(clean_files)} files")


# ============================================
# 2️⃣ صحح مشكلة الـ WL اللي مش اتشالت
# ============================================
def has_workload(filename):
    """
    بيكشف كل أنواع الـ Workload في اسم الملف
    """
    patterns = [
        r'\+\s*WL',        # +WL أو + WL
        r'\+\s*wl',        # lowercase
        r'Workload',       # كلمة كاملة
        r'workload',       # lowercase
        r'\+WL\.',         # +WL.
        r'WL\.',           # WL.
    ]
    for pattern in patterns:
        if re.search(pattern, filename, re.IGNORECASE):
            return True
    return False


# شيل الـ WL اللي فاتنا
before_wl = len(clean_files)
clean_files = [
    f for f in clean_files 
    if not has_workload(f['filename'])
]
after_wl = len(clean_files)

print(f"\n✅ WL Fix:")
print(f"   قبل: {before_wl}")
print(f"   اتشالوا: {before_wl - after_wl}")
print(f"   بعد:  {after_wl}")


# ============================================
# 3️⃣ صحح التصنيفات الغلط
# ============================================

# تصنيفات محسّنة ومفصّلة أكتر
BETTER_CATEGORIES = {
    "Scheduling": [
        "scheduling", "scheduler", "time slot", "timeslot",
        "time interval", "automatic scheduling", "auto scheduling",
        "cs scheduling", "fo scheduling"
    ],
    "Maintenance": [
        "maintenance", "bpd", "technical maintenance",
        "ftth maintenance", "ont", "ont status",
        "cabinet task", "data maintenance", "cables task"
    ],
    "Installation": [
        "installation", "reserve activation", "reservation",
        "migration", "subscriber", "survey",
        "no free port", "ftth migration", "priority flag"
    ],
    "Service_Activation": [
        "service activation", " sa ", "cross-connection",
        "wrong card", "wrong port", "port amendment",
        "vc4", "modify sa", "sa post production",
        "remove reservation activity", "damaged port",
        "port technology", "linetype", "msan"
    ],
    "Dashboard_Reports": [
        "dashboard", "kpi", "report", "reporting",
        "monitoring", "performance", "raw data",
        "final score", "frs", "batch"
    ],
    "Security": [
        "security", "authentication", "quota",
        "session", "data leakage", "vulnerability",
        "two factor", "restrict"
    ],
    "User_Management": [
        "login", "user manual", "mobile app",
        "mobile application", "hiveworx", "desktop user",
        "team leader mobile", "bulk upload", "hr bulk"
    ],
    "Device_Management": [
        "cpe", "sold devices", "installment",
        "rent", "device", "cpe offer"
    ],
    "Network": [
        "odn", "network element", "zone network",
        "ftth portal", "network type", "bpm",
        "high availability", "fvno", "oss"
    ],
    "Work_Orders": [
        "work order", "enterprise work", "wcap",
        "reopen", "pending assigned", "temp assigned",
        "serials", "dispatch score", "physical cases",
        "qr code", "customer care portal",
        "customer segment", "exported sheets"
    ],
    "Other": []
}


def better_categorize(filename, folder_path):
    """
    تصنيف محسّن بيبحث في اسم الملف والفولدر
    """
    # دمج الاسمين للبحث فيهم
    search_text = (filename + " " + folder_path).lower()
    
    best_category = "Other"
    best_score = 0
    
    for category, keywords in BETTER_CATEGORIES.items():
        if category == "Other":
            continue
        
        score = 0
        for keyword in keywords:
            if keyword.lower() in search_text:
                score += 1
        
        if score > best_score:
            best_score = score
            best_category = category
    
    return best_category


# طبّق التصنيف الجديد
old_categories = defaultdict(int)
new_categories = defaultdict(int)

for file_info in clean_files:
    old_cat = file_info['category']
    new_cat = better_categorize(
        file_info['filename'], 
        file_info['folder']
    )
    
    old_categories[old_cat] += 1
    new_categories[new_cat] += 1
    
    # حدّث التصنيف
    file_info['category'] = new_cat


print(f"\n✅ Category Fix:")
print(f"{'Category':<20} {'Old':>6} {'New':>6}")
print("-" * 35)
all_cats = set(list(old_categories.keys()) + 
               list(new_categories.keys()))
for cat in sorted(all_cats):
    old = old_categories.get(cat, 0)
    new = new_categories.get(cat, 0)
    change = "✅" if new != old else "  "
    print(f"{change} {cat:<20} {old:>6} {new:>6}")


# ============================================
# 4️⃣ شيل التكرار
# ============================================

def get_version_score(version_str):
    """
    بيحوّل الـ version لرقم للمقارنة
    v0.3 → 3
    v1.0 → 100
    V2.0 → 200
    """
    if not version_str or version_str == "Unknown":
        return -1
    
    match = re.search(r'(\d+)\.(\d+)', version_str)
    if match:
        major = int(match.group(1))
        minor = int(match.group(2))
        return major * 100 + minor
    return -1


def remove_duplicates(files):
    """
    بيشيل التكرار بطريقة ذكية
    """
    
    # =====================================
    # Case 1: نفس اسم الملف في أماكن مختلفة
    # =====================================
    filename_groups = defaultdict(list)
    
    for f in files:
        # مفتاح = اسم الملف بدون أرقام زيادة زي (1) (2) (3)
        clean_name = re.sub(r'\s*\(\d+\)\s*', '', f['filename'])
        clean_name = clean_name.lower().strip()
        filename_groups[clean_name].append(f)
    
    deduplicated = []
    removed_count = 0
    removal_log = []
    
    for clean_name, group in filename_groups.items():
        if len(group) == 1:
            # ملف واحد بس → خده
            deduplicated.append(group[0])
        else:
            # أكتر من ملف بنفس الاسم
            best = choose_best_file(group)
            deduplicated.append(best)
            
            for f in group:
                if f != best:
                    removed_count += 1
                    removal_log.append({
                        "removed_file": f['filename'],
                        "removed_from": f['folder'],
                        "kept_file": best['filename'],
                        "kept_in": best['folder'],
                        "reason": "Duplicate filename"
                    })
    
    # =====================================
    # Case 2: نفس الـ CR + نفس نوع الوثيقة
    #          بس versions مختلفة
    # =====================================
    cr_doc_groups = defaultdict(list)
    
    for f in deduplicated:
        cr = f['cr_number']
        doc_type = f['type']
        
        if cr != "Unknown":
            key = f"{cr}|{doc_type}"
            cr_doc_groups[key].append(f)
    
    final_files = []
    seen_keys = set()
    
    for f in deduplicated:
        cr = f['cr_number']
        doc_type = f['type']
        
        if cr == "Unknown":
            final_files.append(f)
            continue
        
        key = f"{cr}|{doc_type}"
        
        if key not in seen_keys:
            seen_keys.add(key)
            group = cr_doc_groups[key]
            
            if len(group) > 1:
                best = choose_best_file(group)
                final_files.append(best)
                
                for f2 in group:
                    if f2 != best:
                        removed_count += 1
                        removal_log.append({
                            "removed_file": f2['filename'],
                            "removed_from": f2['folder'],
                            "kept_file": best['filename'],
                            "kept_in": best['folder'],
                            "reason": f"Lower version ({f2['version']} < {best['version']})"
                        })
            else:
                final_files.append(group[0])
    
    return final_files, removed_count, removal_log


def choose_best_file(group):
    """
    من مجموعة ملفات متشابهة، اختار الأفضل
    """
    
    def score(f):
        points = 0
        
        # ✅ نقطة للـ version الأعلى
        points += get_version_score(f['version']) * 10
        
        # ✅ نقطة لو في فولدر CRs (أكثر تنظيماً)
        if f['folder'].startswith('CRs\\'):
            points += 5
        
        # ✅ نقطة لو مش فيه أرقام زيادة في الاسم
        if not re.search(r'\(\d+\)', f['filename']):
            points += 3
        
        # ❌ نقطة ناقص لو اسمه طويل جداً (غالباً نسخة بديلة)
        points -= len(f['filename']) * 0.01
        
        return points
    
    return max(group, key=score)


# طبّق إزالة التكرار
final_files, removed, removal_log = remove_duplicates(clean_files)

print(f"\n✅ Duplicate Removal:")
print(f"   قبل:      {len(clean_files)}")
print(f"   اتشالوا:  {removed}")
print(f"   النهائي:  {len(final_files)}")


# ============================================
# 5️⃣ إحصائيات نهائية
# ============================================
final_categories = defaultdict(list)
for f in final_files:
    final_categories[f['category']].append(f)

print(f"\n📊 Final Categories:")
print("-" * 35)
for cat, files in sorted(final_categories.items(), 
                          key=lambda x: -len(x[1])):
    print(f"   {cat:<22} {len(files):>3} files")


# ============================================
# 6️⃣ حفظ النتائج
# ============================================
os.makedirs("cleaned", exist_ok=True)

# الملفات النهائية
with open("cleaned/final_files.json", "w", 
          encoding="utf-8") as f:
    json.dump(final_files, f, 
              ensure_ascii=False, indent=2)

# سجل التكرار اللي اتشال
with open("cleaned/removed_duplicates.json", "w", 
          encoding="utf-8") as f:
    json.dump(removal_log, f, 
              ensure_ascii=False, indent=2)

# تقرير نهائي
with open("cleaned/final_report.txt", "w", 
          encoding="utf-8") as f:
    
    f.write("=" * 60 + "\n")
    f.write("         FINAL CLEANED FILES REPORT\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("📊 SUMMARY:\n")
    f.write("-" * 40 + "\n")
    f.write(f"Original files:    {len(clean_files)}\n")
    f.write(f"WL removed:        {before_wl - after_wl}\n")
    f.write(f"Duplicates removed:{removed}\n")
    f.write(f"Final clean files: {len(final_files)}\n\n")
    
    f.write("🏷️ BY CATEGORY:\n")
    f.write("-" * 40 + "\n")
    for cat, files in sorted(final_categories.items(),
                              key=lambda x: -len(x[1])):
        f.write(f"\n📁 {cat} ({len(files)} files):\n")
        for file in files:
            f.write(f"   📄 {file['filename']}\n")
            f.write(f"      CR#:     {file['cr_number']}\n")
            f.write(f"      Type:    {file['type']}\n")
            f.write(f"      Version: {file['version']}\n")
            f.write(f"      Folder:  {file['folder']}\n\n")
    
    f.write("\n\n🗑️ REMOVED DUPLICATES LOG:\n")
    f.write("-" * 40 + "\n")
    for log in removal_log:
        f.write(f"Removed: {log['removed_file']}\n")
        f.write(f"From:    {log['removed_from']}\n")
        f.write(f"Kept:    {log['kept_file']}\n")
        f.write(f"Reason:  {log['reason']}\n\n")

print("\n✅ Saved in 'cleaned' folder:")
print("   📄 final_files.json")
print("   📄 removed_duplicates.json")
print("   📄 final_report.txt")

📂 Loaded: 236 files

✅ WL Fix:
   قبل: 236
   اتشالوا: 2
   بعد:  234

✅ Category Fix:
Category                Old    New
-----------------------------------
✅ Dashboard_Reports        12     33
   Device_Management         9      9
✅ Installation             30     36
✅ Maintenance              31     32
✅ Network                   0     11
✅ Other                    31      4
✅ Scheduling               75     58
✅ Service_Activation       34     17
✅ User_Management          12     13
✅ Work_Orders               0     21

✅ Duplicate Removal:
   قبل:      234
   اتشالوا:  91
   النهائي:  143

📊 Final Categories:
-----------------------------------
   Scheduling              28 files
   Maintenance             27 files
   Installation            22 files
   Dashboard_Reports       16 files
   Work_Orders             16 files
   Service_Activation       9 files
   User_Management          9 files
   Network                  8 files
   Other                    4 files
   Device_Manageme

In [5]:
import os
import json
import shutil
from pathlib import Path
from collections import defaultdict

# ============================================
# 1️⃣ لود الملفات النهائية
# ============================================
with open("cleaned/final_files.json", "r", 
          encoding="utf-8") as f:
    final_files = json.load(f)

print(f"✅ Loaded: {len(final_files)} files")

# ============================================
# 2️⃣ المسارات
# ============================================
# 🔴 غيّر المسار ده
ROOT_PATH = r"WFM"

# الفولدر الجديد المنظم
OUTPUT_PATH = r"WFM_CLEAN"

# ============================================
# 3️⃣ إنشاء الهيكل المنظم
# ============================================
def create_organized_copy(final_files, root_path, output_path):
    
    # إحصائيات
    stats = {
        "copied": 0,
        "failed": 0,
        "skipped": 0
    }
    
    errors = []
    copied_files = []
    
    print(f"\n📁 Creating organized structure in:")
    print(f"   {output_path}\n")
    
    # ----------------------------------------
    # امشي على كل ملف
    # ----------------------------------------
    for file_info in final_files:
        
        filename  = file_info["filename"]
        folder    = file_info["folder"]
        category  = file_info["category"]
        cr_number = file_info["cr_number"]
        doc_type  = file_info["type"]
        
        # ----------------------------------------
        # المسار الأصلي
        # ----------------------------------------
        src = os.path.join(root_path, folder, filename)
        
        # ----------------------------------------
        # المسار الجديد المنظم
        # ----------------------------------------
        # الهيكل:
        # WFM_CLEAN/
        # └── Category/
        #     └── CR_Number/
        #         └── Doc_Type/
        #             └── filename
        
        if cr_number == "Unknown":
            cr_folder = "General"
        else:
            cr_folder = cr_number
            
        dst_folder = os.path.join(
            output_path,
            category,
            cr_folder,
            doc_type
        )
        
        dst = os.path.join(dst_folder, filename)
        
        # ----------------------------------------
        # تحقق من وجود الملف الأصلي
        # ----------------------------------------
        if not os.path.exists(src):
            stats["skipped"] += 1
            errors.append({
                "file": filename,
                "error": "Source not found",
                "src": src
            })
            continue
        
        # ----------------------------------------
        # إنشاء الفولدر الجديد
        # ----------------------------------------
        os.makedirs(dst_folder, exist_ok=True)
        
        # ----------------------------------------
        # نسخ الملف
        # ----------------------------------------
        try:
            shutil.copy2(src, dst)
            stats["copied"] += 1
            
            copied_files.append({
                "filename": filename,
                "from": src,
                "to": dst,
                "category": category,
                "cr_number": cr_number,
                "doc_type": doc_type
            })
            
            print(f"✅ {filename[:50]}")
            
        except Exception as e:
            stats["failed"] += 1
            errors.append({
                "file": filename,
                "error": str(e),
                "src": src
            })
            print(f"❌ {filename[:50]}")
    
    return stats, errors, copied_files


# ============================================
# 4️⃣ تشغيل النسخ
# ============================================
stats, errors, copied_files = create_organized_copy(
    final_files, 
    ROOT_PATH, 
    OUTPUT_PATH
)

# ============================================
# 5️⃣ طباعة النتائج
# ============================================
print(f"\n{'='*50}")
print(f"📊 COPY RESULTS:")
print(f"{'='*50}")
print(f"✅ Copied:  {stats['copied']}")
print(f"⚠️  Skipped: {stats['skipped']}")
print(f"❌ Failed:  {stats['failed']}")

# ============================================
# 6️⃣ عرض الهيكل الجديد
# ============================================
print(f"\n📁 NEW STRUCTURE:")
print(f"{'='*50}")

category_count = defaultdict(lambda: defaultdict(int))

for f in copied_files:
    category_count[f['category']][f['doc_type']] += 1

for category, doc_types in sorted(category_count.items()):
    total = sum(doc_types.values())
    print(f"\n📂 {category}/ ({total} files)")
    for doc_type, count in sorted(doc_types.items()):
        print(f"   📂 {doc_type}/ ({count} files)")

# ============================================
# 7️⃣ حفظ التقرير
# ============================================
os.makedirs("cleaned", exist_ok=True)

# تقرير النسخ
report = {
    "stats": stats,
    "structure": {
        cat: dict(types) 
        for cat, types in category_count.items()
    },
    "errors": errors
}

with open("cleaned/copy_report.json", "w", 
          encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# تقرير نصي
with open("cleaned/copy_report.txt", "w", 
          encoding="utf-8") as f:
    
    f.write("=" * 60 + "\n")
    f.write("      ORGANIZED COPY REPORT\n")
    f.write("=" * 60 + "\n\n")
    
    f.write("📊 STATS:\n")
    f.write("-" * 40 + "\n")
    f.write(f"✅ Copied:  {stats['copied']}\n")
    f.write(f"⚠️  Skipped: {stats['skipped']}\n")
    f.write(f"❌ Failed:  {stats['failed']}\n\n")
    
    f.write("📁 STRUCTURE:\n")
    f.write("-" * 40 + "\n")
    
    for category, doc_types in sorted(category_count.items()):
        total = sum(doc_types.values())
        f.write(f"\n📂 {category}/ ({total} files)\n")
        for doc_type, count in sorted(doc_types.items()):
            f.write(f"   📂 {doc_type}/ ({count} files)\n")
    
    if errors:
        f.write("\n\n❌ ERRORS:\n")
        f.write("-" * 40 + "\n")
        for err in errors:
            f.write(f"File:  {err['file']}\n")
            f.write(f"Error: {err['error']}\n\n")

print(f"\n✅ Reports saved:")
print(f"   📄 cleaned/copy_report.json")
print(f"   📄 cleaned/copy_report.txt")
print(f"\n📁 Organized files in:")
print(f"   {OUTPUT_PATH}")

✅ Loaded: 143 files

📁 Creating organized structure in:
   WFM_CLEAN

✅ FCC Operations Monitoring Dashboard Batch 2 (v0.1)
✅ WFM Operations Monitor Dashboard Batch 2 CR (v0.1)
✅ (CR-2024-136)WFM BOX Complain CR (V0.1).docx
✅ WFM - SA Post Production - Modify SA APIs CR v3 -.
✅ (CR-2024-170) WFM Detect Enterprise Work Orders CR
✅ FTTH_Portal_Add_ODN_network_type_ATPs(v1.0).docx
✅ ODN IOT TC(V1.0).xlsx
✅ (CR-2025-202)_FTTH_Portal_Add_ODN_network_type_CR.
✅ WFM Detect Enterprise Migration CR ATPs (v1.0).doc
✅ (CR-2025-203) WFM Detect Enterprise Migration CR (
✅ WFM_Remove Reservation Activity-damaged port.docx
✅ CR-2025-209 WFM_Enterprise_Maintenance_(v0.5).docx
✅ WFM - Desktop Users Login Modification Amendment A
✅ WFM - Desktop Users Login Modification Amendment v
✅ CR-2025-213  Avail fail scenario in data maintenan
✅ CR-2025-214   Remove permission for all FME to clo
✅ Add new Parameters to FO Raw Data Reports v(0.2).d
✅ FO_Team Leader KPI_FInal_Score_Report_ ATPs v1.0.d
✅ CR-2025-218 

In [ ]:
المرحلة 1: قراءة الملفات
━━━━━━━━━━━━━━━━━━━━━━━━
كل الملفات: مئات
↓
folder_structure.json ✅

المرحلة 2: تنظيف وتحليل
━━━━━━━━━━━━━━━━━━━━━━━━
شلنا: temp + WL + svn
فضل: 236 ملف نظيف
↓
clean_files.json ✅

المرحلة 3: تصحيح التصنيف
━━━━━━━━━━━━━━━━━━━━━━━━
10 categories منظمة
↓
categories محسّنة ✅

المرحلة 4: شيل التكرار
━━━━━━━━━━━━━━━━━━━━━━━━
شلنا: 91 ملف مكرر
فضل: 143 ملف
↓
final_files.json ✅

المرحلة 5: النسخة المنظمة
━━━━━━━━━━━━━━━━━━━━━━━━
143 ملف في هيكل منظم
↓
WFM_CLEAN/ ✅